# Data Representation and Preprocessing

## Data Representation

A bag-of-words representation was used to represent abstract text in this classifier. In this approach, each abstract is treated as an unordered collection of words, where each word is considered an independent feature with its frequency serving as the feature value. This representation was selected for its simplicity and computational efficiency.

For each class, the data representation consists of:

- **Word Frequency Counts** - The number of occurrences of each word within documents of that class
- **Total Word Count** - The total number of words observed across all documents in the class
- **Vocabulary** - A set of all unique words appearing in the training data for each class

### Preprocessing

In the standard Naive Bayes model abstracts were split by whitespace only. No further preprocessing was applied, meaning punctuation and original casing were preserved with all words retained. The model was trained directly on this representation.

This simple preprocessing approach was chosen because Naive Bayes performs surprisingly well with minimal preprocessing. As a baseline model, it makes sense to avoid heavy preprocessing and leave that for the improved model to provide a clean comparison point.

### Standard Naive Bayes Implementation

The classifier was implemented as a Multinomial Naive Bayes model that learns from feature distributions of each class from the training data.

The implementation calculates and learns the prior probabilities and likelihoods of features for each class, then uses this information to make predictions on unseen abstracts, predicting the class based on features.

**Calculating Prior probabilites**
Prior Probabilities estimate the likelihood of each class before observing any words. These were calculated using:

$$P(c) = \frac{\text{count}(c)}{N}$$

where $\text{count}(c)$ is the number of training documents in class $c$, and $N$ is the total number of training documents.

**Calculating Likeihoods**

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total words in } c + |V|}$$

Where $\text{count}(w, c)$ is the number of times word $w$ appears in class $c$, and $|V|$ is the vocabulary size across all classes

Laplace smoothing was added in the baseline model as many words in the test documents may not have been seen in training so there was a risk of producing zero probablities during predictions of abstracts containing unseen words. To resolve this, every word's count is incremented by 1 before computing the likelihood, and the vocabulary size is added to the denominator to ensure the distribution remains valid. This guarantees that no word ever receives a probability of zero, regardless of whether it appeared in training.

**Prediction Phase**

For prediction, the classifier computes a score for each class using log probabilities, this because multiplying many small probabilities can cause numerical underflow.
the score consists of the prior probability each class along with the sum of all log likihoods of each word in the class. 

$$\text{score}(c) = \log P(c) + \sum_{i=1}^{n} \log P(w_i|c)$$

where
- $P(c)$ is the prior probability of class $c$
- $P(w_i|c)$ is the likelihood of word $w_i$ in class $c$
- The sum is over all words in the test document

the resulting prediction would be the highest scoring class.



# Model Improvements

## Improvements Implemented

To improve upon the Standard Naive Bayes classifier, the following enhancements were tested:

### 1. Stronger Data Preprocessing
- **Stopword removal** - Removes common words that appear across nearly all documents
- **Punctuation removal and Lowercasing** - Strips punctuation and normalizes all text to lowercase to ensure differently cased versions of the same word are seen as identical. 

### 2. Compound Word Merging
Merges known compound words (e.g., "homo sapiens" → "homo_sapiens") that would otherwise be counted as two independent features, allowing the model to recognize them as a single feature

### 3. Vocabulary Filtering by Minimum Frequency
Applies a minimum document frequency threshold to eliminate rare words that appear in fewer than MIN_FREQ documents.


## How it was implemented
A new method called preproccess was created with the abstract text as a parameter. This method performs punctuation removal and lowercasing and contains the implementation of stopword removal and compound word merging. It returns the processed text.

A list of all known stopwords was added with a new hyperparameter "USE_STOPWORDS" assigned a boolean value to enable or disable stopword removal within the preprocess method.

A list of all known compound words was also added with the hyperparameter "USE_COMPOUND_MERGES" with a boolean value to enable or disable compound merges within the preprocess method.

Minimum frequency filtering was implemented by first counting how many training documents contain each feature, then keeping only features that appear at least "MIN_FREQ" times. 

## How hyperparameters were tuned
### Standard Model
The baseline model did not introduce new preprocessing hyperparameters. It uses whitespace tokenization only, no stopword removal, compound merging and minimum frequency filter

### Improved Model
Four hyperparameters were introduced:
- `NGRAM_SIZE = 1`
- `MIN_FREQ = 2`
- `USE_STOPWORDS = True`
- `USE_COMPOUND_MERGES True`

### How They Were Set/Tuned
Hyperparameters were tuned with controlled experiments by changing one setting at a time and comparing validation accuracy on the same 80/20 split:
- `NGRAM_SIZE`: tested with unigrams 1 vs larger n-grams, unigram performed best and was retained
- `MIN_FREQ`: tested multiple thresholds, 2 gave the best balance between removing noise and keeping useful terms
- `USE_STOPWORDS`: enabled because it consistently improved accuracy by removing non informative words
- `USE_COMPOUND_MERGES`: enabled, results did not change in the test set.

# Evaluation Procedure

### Train/Test Split
- **Split Ratio** - 80% training / 20% test
- **Stratification** - The split maintains the original class distribution in both sets
- **Random Seed** - Fixed random seed (42) ensures reproducibility

### Evaluation Metric

Both the baseline and improved models were evaluated using the same methodology to ensure fair comparison,
accuracy was used as the primary evaluation metric and represents the proportion of test abstracts correctly classified by each model.

### Comparison

Both models are trained on the same 80% training subset and evaluated on the identical 20% test subset, allowing direct comparison of their predictive performance.

## Results

To evaluate both models, the training data was split into an 80/20 stratified
train/validation split. A majority-class baseline was also computed

| Model | Train Split | Val Accuracy | Kaggle (Public) |
|---|---|---|---|
| Majority class (baseline) | — | ~0.536 | — |
| Standard Naive Bayes | 3,200 docs | 0.9413 | — |
| Improved Naive Bayes | 3,200 docs | 0.9463 | 0.976 |

**Standard Naive Bayes** achieved a validation accuracy of 0.9413, substantially
above the majority-class baseline of approximately 0.536. This confirms that the
standard model is learning genuinely
discriminative signal from the abstracts. The model uses unigram bag-of-words
features with Laplace smoothing and log-space computation, trained on raw token
counts after lowercasing and punctuation removal.

**The improved model** achieved a validation accuracy of 0.9463, a modest but
consistent improvement over the standard model (+0.005). The improvements —
expanded stopword removal, MIN_FREQ vocabulary filtering, and merging of known
multi-word biological entities such as *Escherichia coli* and abbreviated forms
like *e coli* reduce noise in the feature space and prevent double-counting of
correlated evidence. For instance, without entity merging, the words "escherichia"
and "coli" are treated as two independent pieces of evidence for the Bacteria
class when they represent a single organism, inflating the model's confidence.

The improved model's Kaggle public leaderboard score of **0.976** is notably
higher than its validation accuracy of 0.9463. This is expected: the Kaggle
submission retrains on the full 4,000-document training set rather than the
3,200-document training split, giving the model 25% more data. The additional
training examples improve generalisation, particularly for the minority classes
Archaea (128 docs) and Virus (126 docs), which have limited training signal in
the 80% split.

Both models substantially outperform the majority-class baseline, and the improved
model outperforms the standard model on both validation and test data, confirming
that the preprocessing enhancements generalise beyond the training distribution.

### Standard bayes model

In [34]:
from collections import defaultdict
import pandas as pd
import math

def calculate_likelihood(class_label, word):
    return (word_counts[class_label][word] + 1) / (total_words_per_class[class_label] + len(vocab))

training_df = pd.read_csv("trg.csv", header=0)
test_df = pd.read_csv("tst.csv", header=0)
class_counts = training_df["class"].value_counts()

prior_probabilities = {
    c: count / training_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

likelihoods = {c: defaultdict(int) for c in class_counts.keys()}

vocab = set()

for _, row in training_df.iterrows():
    class_label = row["class"]
    words = row["abstract"].split()

    for word in words:
        word_counts[class_label][word] += 1
        total_words_per_class[class_label] += 1
        vocab.add(word)

def predict(abstract):
    words = abstract.split()
    scores = {}

    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])

        for word in words:
            prob = calculate_likelihood(c, word)
            score += math.log(prob)

        scores[c] = score

    return max(scores, key=scores.get)

predicted = test_df["abstract"].apply(predict)

submission = pd.DataFrame({
    "id": test_df["id"],
    "class": predicted
})
submission.to_csv("submission.csv", index=False)
print(submission)



       id class
0       1     B
1       2     E
2       3     E
3       4     E
4       5     E
..    ...   ...
995   996     B
996   997     E
997   998     E
998   999     A
999  1000     E

[1000 rows x 2 columns]


### Improved bayes model

In [11]:
from collections import defaultdict
import pandas as pd
import math
import re

NGRAM_SIZE = 1
MIN_FREQ = 2
USE_STOPWORDS = True
USE_COMPOUND_MERGES = False

COMPOUNDS = [
    ("homo", "sapiens"),
    ("escherichia", "coli"),
    ("saccharomyces", "cerevisiae"),
    ("caenorhabditis", "elegans"),
    ("drosophila", "melanogaster"),
    ("arabidopsis", "thaliana"),
    ("bacillus", "subtilis"),
    ("mycobacterium", "tuberculosis"),
    ("helicobacter", "pylori"),
    ("haemophilus", "influenzae"),
    ("pseudomonas", "aeruginosa"),
    ("staphylococcus", "aureus"),
    ("methanococcus", "jannaschii"),
    ("sulfolobus", "solfataricus"),
    ("archaeoglobus", "fulgidus"),
    ("thermoplasma", "acidophilum"),
    ("pyrococcus", "horikoshii"),
    ("streptomyces", "coelicolor"),
]

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "in", "to", "is", "are", "was",
    "were", "be", "been", "has", "have", "had", "it", "its", "for", "on",
    "at", "by", "as", "with", "this", "that", "these", "those", "from",
    "not", "but", "we", "our", "their", "they", "which", "also", "into",
    "can", "more", "than", "two", "three", "one", "between", "among",
    "both", "all", "each", "such", "other", "within", "through", "after",
    "before", "during", "over", "under", "up", "out", "no", "only", "so",
    "if", "when", "there", "here", "how", "what", "who", "about", "while",
    "since", "used", "using", "show", "showed", "shown", "found", "may",
    "could", "would", "should", "will", "do", "did", "does",
}


def preprocess(text):
    text = text.lower()
    # always strip punctuation/digits
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()

    # merge known compounds BEFORE stopword removal so pairs remain adjacent
    if USE_COMPOUND_MERGES:
        merged = []
        i = 0
        while i < len(tokens):
            matched = False
            if i < len(tokens) - 1:
                for w1, w2 in COMPOUNDS:
                    if tokens[i] == w1 and tokens[i + 1] == w2:
                        merged.append(f"{w1}_{w2}")
                        i += 2
                        matched = True
                        break
            if not matched:
                merged.append(tokens[i])
                i += 1
        tokens = merged

    # remove stopwords + short tokens (configurable)
    if USE_STOPWORDS:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    else:
        tokens = [t for t in tokens if len(t) > 1]

    return tokens


def generate_ngrams(words, n):
    if n <= 1:
        return []
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append("_".join(words[i:i + n]))
    return ngrams


def calculate_likelihood(class_label, word):
    tf = word_counts[class_label][word]
    return (tf + 1) / (total_words_per_class[class_label] + len(filtered_vocab))


training_df = pd.read_csv("trg.csv", header=0)
test_df = pd.read_csv("tst.csv", header=0)
class_counts = training_df["class"].value_counts()

prior_probabilities = {
    c: count / training_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

# Build vocab and document frequencies
vocab = set()
doc_count = defaultdict(int)
N = training_df.shape[0]
for _, row in training_df.iterrows():
    token_list = preprocess(row["abstract"])
    features = set(token_list)
    # include n-grams for document frequency using token order
    features.update(generate_ngrams(token_list, NGRAM_SIZE))
    for f in features:
        doc_count[f] += 1
    vocab.update(features)

# Apply MIN_FREQ filtering
filtered_vocab = {w for w, cnt in doc_count.items() if cnt >= MIN_FREQ}

# Second pass: populate class word counts limited to filtered_vocab
for _, row in training_df.iterrows():
    class_label = row["class"]
    features = preprocess(row["abstract"])
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]
    for f in features:
        word_counts[class_label][f] += 1
        total_words_per_class[class_label] += 1


def predict(abstract):
    features = preprocess(abstract)
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]

    if not features:
        return max(prior_probabilities, key=prior_probabilities.get)

    word_freq = defaultdict(int)
    for f in features:
        word_freq[f] += 1

    scores = {}
    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])
        for f, cnt in word_freq.items():
            score += cnt * math.log(calculate_likelihood(c, f))
        scores[c] = score

    return max(scores, key=scores.get)


predicted = test_df["abstract"].apply(predict)

submission = pd.DataFrame({
    "id": test_df["id"],
    "class": predicted
})
submission.to_csv("submission.csv", index=False)
print(submission)

       id class
0       1     B
1       2     E
2       3     E
3       4     E
4       5     E
..    ...   ...
995   996     B
996   997     E
997   998     E
998   999     A
999  1000     E

[1000 rows x 2 columns]


## Test/train evaluation models

80/20 train/test split
Standard model, followed by improved model.

In [2]:
from collections import defaultdict
from sklearn.model_selection import train_test_split
import pandas as pd
import math

full_df = pd.read_csv("trg.csv", header=0)

# Implement 80/20 train/test split
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df["class"])

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

class_counts = train_df["class"].value_counts()

prior_probabilities = {
    c: count / train_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

vocab = set()

# Training phase
for _, row in train_df.iterrows():
    class_label = row["class"]
    words = row["abstract"].split()

    for word in words:
        word_counts[class_label][word] += 1
        total_words_per_class[class_label] += 1
        vocab.add(word)

def calculate_likelihood(class_label, word):
    return (word_counts[class_label][word] + 1) / (total_words_per_class[class_label] + len(vocab))

def predict(abstract):
    words = abstract.split()
    scores = {}

    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])

        for word in words:
            prob = calculate_likelihood(c, word)
            score += math.log(prob)

        scores[c] = score

    return max(scores, key=scores.get)

# Testing phase
predicted = test_df["abstract"].apply(predict)
accuracy = (predicted == test_df["class"]).sum() / len(test_df)
print(f"\nStandard Model Accuracy: {accuracy:.4f}")


Training set size: 3200
Test set size: 800

Standard Model Accuracy: 0.9413


In [8]:
from collections import defaultdict
from sklearn.model_selection import train_test_split
import pandas as pd
import math
import re

full_df = pd.read_csv("trg.csv", header=0)

# Implement 80/20 train/test split
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df["class"])

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

NGRAM_SIZE = 1
MIN_FREQ = 2
USE_STOPWORDS = True
USE_COMPOUND_MERGES = True

COMPOUNDS = [
    ("homo", "sapiens"),
    ("escherichia", "coli"),
    ("saccharomyces", "cerevisiae"),
    ("caenorhabditis", "elegans"),
    ("drosophila", "melanogaster"),
    ("arabidopsis", "thaliana"),
    ("bacillus", "subtilis"),
    ("mycobacterium", "tuberculosis"),
    ("helicobacter", "pylori"),
    ("haemophilus", "influenzae"),
    ("pseudomonas", "aeruginosa"),
    ("staphylococcus", "aureus"),
    ("methanococcus", "jannaschii"),
    ("sulfolobus", "solfataricus"),
    ("archaeoglobus", "fulgidus"),
    ("thermoplasma", "acidophilum"),
    ("pyrococcus", "horikoshii"),
    ("streptomyces", "coelicolor"),
]

STOPWORDS = {   
    "a", "an", "the", "and", "or", "of", "in", "to", "is", "are", "was",
    "were", "be", "been", "has", "have", "had", "it", "its", "for", "on",
    "at", "by", "as", "with", "this", "that", "these", "those", "from",
    "not", "but", "we", "our", "their", "they", "which", "also", "into",
    "can", "more", "than", "two", "three", "one", "between", "among",
    "both", "all", "each", "such", "other", "within", "through", "after",
    "before", "during", "over", "under", "up", "out", "no", "only", "so",
    "if", "when", "there", "here", "how", "what", "who", "about", "while",
    "since", "used", "using", "show", "showed", "shown", "found", "may",
    "could", "would", "should", "will", "do", "did", "does",
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()

    if USE_COMPOUND_MERGES:
        merged = []
        i = 0
        while i < len(tokens):
            matched = False
            if i < len(tokens) - 1:
                for w1, w2 in COMPOUNDS:
                    if tokens[i] == w1 and tokens[i + 1] == w2:
                        merged.append(f"{w1}_{w2}")
                        i += 2
                        matched = True
                        break
            if not matched:
                merged.append(tokens[i])
                i += 1
        tokens = merged

    if USE_STOPWORDS:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    else:
        tokens = [t for t in tokens if len(t) > 1]

    return tokens

def generate_ngrams(words, n):
    if n <= 1:
        return []
    ngrams = []
    for i in range(len(words) - n + 1):
        ngrams.append("_".join(words[i:i + n]))
    return ngrams

# Improved Naive Bayes Model with preprocessing and filtering
class_counts = train_df["class"].value_counts()

prior_probabilities = {
    c: count / train_df.shape[0]
    for c, count in class_counts.items()
}

word_counts = {c: defaultdict(int) for c in class_counts.keys()}
total_words_per_class = defaultdict(int)

# Build vocab and document frequencies
vocab = set()
doc_count = defaultdict(int)
N = train_df.shape[0]

for _, row in train_df.iterrows():
    token_list = preprocess(row["abstract"])
    features = set(token_list)
    features.update(generate_ngrams(token_list, NGRAM_SIZE))
    for f in features:
        doc_count[f] += 1
    vocab.update(features)

# Apply MIN_FREQ filtering
filtered_vocab = {w for w, cnt in doc_count.items() if cnt >= MIN_FREQ}

# Second pass: populate class word counts
for _, row in train_df.iterrows():
    class_label = row["class"]
    features = preprocess(row["abstract"])
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]
    for f in features:
        word_counts[class_label][f] += 1
        total_words_per_class[class_label] += 1

def calculate_likelihood_improved(class_label, word):
    tf = word_counts[class_label][word]
    return (tf + 1) / (total_words_per_class[class_label] + len(filtered_vocab))

def predict_improved(abstract):
    features = preprocess(abstract)
    features.extend(generate_ngrams(features, NGRAM_SIZE))
    features = [f for f in features if f in filtered_vocab]

    if not features:
        return max(prior_probabilities, key=prior_probabilities.get)

    word_freq = defaultdict(int)
    for f in features:
        word_freq[f] += 1

    scores = {}
    for c in class_counts.keys():
        score = math.log(prior_probabilities[c])
        for f, cnt in word_freq.items():
            score += cnt * math.log(calculate_likelihood_improved(c, f))
        scores[c] = score

    return max(scores, key=scores.get)

# Testing phase
predicted = test_df["abstract"].apply(predict_improved)
accuracy = (predicted == test_df["class"]).sum() / len(test_df)
print(f"\nImproved Model Accuracy: {accuracy:.4f}")

Training set size: 3200
Test set size: 800

Improved Model Accuracy: 0.9463
